In [49]:
spark.sql("select * from spark_db.flight_time_clean").show()

+----------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+---------+--------+--------------------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|        CRS_DEP_TIME|            DEP_TIME|           WHEELS_ON|TAXI_IN|        CRS_ARR_TIME|            ARR_TIME|CANCELLED|DISTANCE|              TAX_IN|
+----------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+---------+--------+--------------------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|INTERVAL '11:15' ...|INTERVAL '11:13' ...|INTERVAL '13:43' ...|      5|INTERVAL '14:00' ...|INTERVAL '13:48' ...|        0|     946|INTERVAL '05' MINUTE|
|2000-01-01|        DL|             1479|   

In [57]:
"""
Read data from flight_time and transform as below

    Rename fl_date to dep_date
    Compute arr_date
    Following fields to represent full timestamp
        crs_dep_time
        dep_time
        crs_arr_time
        arr_time
"""
# from pyspark.sql.functions import selectExpr

flight_time_df = spark.read.table("spark_db.flight_time_clean")
flight_time_1_df = flight_time_df.selectExpr(
    "fl_date as dep_date",
    "to_date(dep_date + dep_time + wheels_on) as arr_date", # cros referencing allowed only for renaming
    "dep_date + crs_dep_time as crs_dep_time",
    "dep_date + dep_time as dep_time",
    "arr_date + crs_arr_time as crs_arr_time",
    "arr_date + arr_time as arr_time"
)
flight_time_1_df.show()
# this only selected the columns in the selectExpr but the remaining columns did not show up

+----------+----------+-------------------+-------------------+-------------------+-------------------+
|  dep_date|  arr_date|       crs_dep_time|           dep_time|       crs_arr_time|           arr_time|
+----------+----------+-------------------+-------------------+-------------------+-------------------+
|2000-01-01|2000-01-02|2000-01-01 11:15:00|2000-01-01 11:13:00|2000-01-02 14:00:00|2000-01-02 13:48:00|
|2000-01-01|2000-01-02|2000-01-01 13:15:00|2000-01-01 13:11:00|2000-01-02 15:59:00|2000-01-02 15:43:00|
|2000-01-01|2000-01-02|2000-01-01 14:15:00|2000-01-01 14:14:00|2000-01-02 17:21:00|2000-01-02 16:51:00|
|2000-01-01|2000-01-02|2000-01-01 17:15:00|2000-01-01 17:20:00|2000-01-02 20:13:00|2000-01-02 20:05:00|
|2000-01-01|2000-01-02|2000-01-01 20:15:00|2000-01-01 20:10:00|2000-01-02 23:00:00|2000-01-02 22:40:00|
|2000-01-01|2000-01-01|2000-01-01 06:50:00|2000-01-01 06:49:00|2000-01-01 09:55:00|2000-01-01 10:03:00|
|2000-01-01|2000-01-02|2000-01-01 14:40:00|2000-01-01 14:46:00|2

In [64]:
# done using withColumns as selectExpr did not return all the columns
from pyspark.sql.functions import expr


flight_time_2_df = flight_time_df.withColumnRenamed("fl_date", "dep_date")\
                                .withColumn("arr_date", expr("to_date(dep_date + dep_time + wheels_on)"))\
                                .withColumns({
                                    "crs_dep_time": expr("dep_date + crs_dep_time"),
                                    "dep_time": expr("dep_date + dep_time"),
                                    "crs_arr_time": expr("arr_date + crs_arr_time"),
                                    "arr_time": expr("arr_date + arr_time")
                                })
# arr_date calculated seperately because used in below calculations
flight_time_2_df.show()

+----------+----------+-----------------+------+----------------+----+--------------+-------------------+-------------------+--------------------+-------+-------------------+-------------------+---------+--------+--------------------+----------+
|  dep_date|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|       crs_dep_time|           dep_time|           WHEELS_ON|TAXI_IN|       crs_arr_time|           arr_time|CANCELLED|DISTANCE|              TAX_IN|  arr_date|
+----------+----------+-----------------+------+----------------+----+--------------+-------------------+-------------------+--------------------+-------+-------------------+-------------------+---------+--------+--------------------+----------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|2000-01-01 11:15:00|2000-01-01 11:13:00|INTERVAL '13:43' ...|      5|2000-01-02 14:00:00|2000-01-02 13:48:00|        0|     946|INTERVAL '05' MINUTE|2000-01-02|
|2000-01-01|    

In [68]:
# using col function to give each column access for the calculations to take place
# in that case the evaluation will not happen using expr

from pyspark.sql.functions import to_date, col

flight_time_3_df = flight_time_df.withColumnRenamed("fl_date", "dep_date")\
                                .withColumn("arr_date", to_date(col("dep_date") + col("dep_time") + col("wheels_on")))\
                                .withColumns({
                                    "crs_dep_time": col("dep_date") + col("crs_dep_time"),
                                    "dep_time": col("dep_date") + col("dep_time"),
                                    "crs_arr_time": col("arr_date") + col("crs_arr_time"),
                                    "arr_time": col("arr_date") + col("arr_time")
                                })
flight_time_3_df.show()

+----------+----------+-----------------+------+----------------+----+--------------+-------------------+-------------------+--------------------+-------+-------------------+-------------------+---------+--------+--------------------+----------+
|  dep_date|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|       crs_dep_time|           dep_time|           WHEELS_ON|TAXI_IN|       crs_arr_time|           arr_time|CANCELLED|DISTANCE|              TAX_IN|  arr_date|
+----------+----------+-----------------+------+----------------+----+--------------+-------------------+-------------------+--------------------+-------+-------------------+-------------------+---------+--------+--------------------+----------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|2000-01-01 11:15:00|2000-01-01 11:13:00|INTERVAL '13:43' ...|      5|2000-01-02 14:00:00|2000-01-02 13:48:00|        0|     946|INTERVAL '05' MINUTE|2000-01-02|
|2000-01-01|    

In [81]:
# using col is better some times becuase it gives access to columm functions instead of using SQL functions like expr
# some times we might not find the SQL function that are needed for our transformations
# the col class has its own spark functions that might not be available as a function in SQL

flight_time_3_df.where((col("op_carrier_fl_num") == 1451) & (col("dep_date") == '2000-01-01')).show()

+----------+----------+-----------------+------+----------------+----+--------------+-------------------+-------------------+--------------------+-------+-------------------+-------------------+---------+--------+--------------------+----------+
|  dep_date|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|       crs_dep_time|           dep_time|           WHEELS_ON|TAXI_IN|       crs_arr_time|           arr_time|CANCELLED|DISTANCE|              TAX_IN|  arr_date|
+----------+----------+-----------------+------+----------------+----+--------------+-------------------+-------------------+--------------------+-------+-------------------+-------------------+---------+--------+--------------------+----------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|2000-01-01 11:15:00|2000-01-01 11:13:00|INTERVAL '13:43' ...|      5|2000-01-02 14:00:00|2000-01-02 13:48:00|        0|     946|INTERVAL '05' MINUTE|2000-01-02|
|2000-01-01|    